# StoryZop: Instagram Story Visual Analysis

This notebook runs the complete StoryZop pipeline on Google Colab.

**Before running:**
1. Change runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Add your Instagram `sessionid` to **Colab Secrets** (🔑 icon on the left sidebar):
   - Name: `INSTAGRAM_SESSIONID`
   - Value: *(your sessionid cookie value)*
   - Toggle: Enable notebook access

## 1. Install Dependencies & Clone Repo

In [ ]:
# Core dependencies
!pip install -q playwright Pillow pydantic pydantic-settings python-dotenv nest-asyncio

# Install Chromium WITH system dependencies (critical for Colab)
!playwright install --with-deps chromium

# AI / Vision dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers>=4.57.0 accelerate qwen-vl-utils bitsandbytes easyocr

# Clone the repo
!git clone https://github.com/10Unknownboy/StoryZop.git 2>/dev/null || echo 'Repo already cloned'
%cd StoryZop

# Enable async in Colab
import nest_asyncio
nest_asyncio.apply()

print('\n✅ All dependencies installed.')

## 2. GPU Check

In [ ]:
from src.vision.gpu import GPUManager

GPUManager.print_gpu_status()

# Check model fit
print(f"\n4B model fits: {GPUManager.estimate_model_fit('4b')}")
print(f"8B model fits: {GPUManager.estimate_model_fit('8b')}")
print(f"32B (4-bit) fits: {GPUManager.estimate_model_fit('32b', quantized=True)}")

## 3. Configuration

The session ID is read securely from Colab Secrets — it is never stored in the notebook.

In [ ]:
from src.config import get_config

# Try reading from Colab Secrets
session_id = None
try:
    from google.colab import userdata
    session_id = userdata.get('INSTAGRAM_SESSIONID')
    print('\u2705 Session ID loaded from Colab Secrets.')
except Exception:
    print('\u26a0\ufe0f Could not read INSTAGRAM_SESSIONID from Colab Secrets.')
    print('Please add it via the \U0001f511 Secrets panel on the left sidebar.')

config = get_config(
    instagram_sessionid=session_id,
    use_4bit_quantization=True,
    headless=True,
)

print(f'Data dir: {config.data_dir}')
print(f'Models: {config.initial_model} | {config.primary_model} | {config.expert_model}')

## 4. Initialize Database

In [ ]:
from src.database.database import Database

db = Database(config.db_path)
db.initialize()

state = db.get_processing_state()
print(f'\u2705 Database ready at {config.db_path}')
print(f'   Completed: {len(state["completed"])} | Pending: {len(state["pending"])} | Incomplete: {len(state["incomplete"])}')

## 5. Load AI Models

Models are loaded eagerly here so you can verify they fit in VRAM before starting the pipeline.

In [ ]:
from src.vision.qwen4b import Qwen4BScreener
from src.vision.qwen8b import Qwen8BAnalyzer
from src.vision.qwen32b import Qwen32BExpert
from src.vision.ocr import OCREngine

# --- 4B Screener (required) ---
print('Loading 4B Screener...')
screener = Qwen4BScreener(config)
screener.load_model()
print('\u2705 4B Screener loaded.\n')

# --- 8B Analyzer (required) ---
print('Loading 8B Analyzer...')
analyzer = Qwen8BAnalyzer(config)
analyzer.load_model()
print('\u2705 8B Analyzer loaded.\n')

# --- 32B Expert (optional, needs high VRAM) ---
expert = None
if GPUManager.estimate_model_fit('32b', quantized=config.use_4bit_quantization):
    print('Loading 32B Expert (4-bit quantized)...')
    expert = Qwen32BExpert(config)
    expert.load_model()
    if expert.is_available:
        print('\u2705 32B Expert loaded.\n')
    else:
        print('\u26a0\ufe0f 32B Expert could not load (insufficient VRAM). Continuing without it.\n')
        expert = None
else:
    print('\u26a0\ufe0f Skipping 32B Expert (not enough VRAM). 8B will handle all analysis.\n')

# --- OCR ---
print('Initializing OCR engine...')
ocr_engine = OCREngine(languages=config.ocr_languages, confidence_threshold=config.ocr_confidence_threshold)
print(f'\u2705 OCR ready (available: {ocr_engine.is_available}).\n')

print('--- All models initialized ---')

## 6. Launch Browser & Authenticate

Launches a headless Chromium browser and injects the Instagram session cookie.

In [ ]:
from src.browser.session import BrowserSession
from src.browser.instagram import InstagramNavigator
from src.browser.stories import StoryNavigator
from src.capture.frame_manager import FrameManager
from src.capture.sampler import StorySampler

# Launch browser
session = BrowserSession(config)
await session.launch()
print('\u2705 Browser launched.')

# Inject session cookie
if config.instagram_sessionid:
    await session.load_sessionid(config.instagram_sessionid)
    print('\u2705 Session ID cookie injected.')
else:
    print('\u26a0\ufe0f No session ID provided. Stories will not be accessible.')

# Create navigators and capture components
instagram_nav = InstagramNavigator(session.page, config)
story_nav = StoryNavigator(session.page, config)
frame_manager = FrameManager(config)
sampler = StorySampler(config, frame_manager)

# Navigate to Instagram and verify login
await instagram_nav.navigate_to_instagram()
is_auth = await instagram_nav.verify_authentication()
print(f'\nAuthenticated: {is_auth}')

if is_auth:
    await instagram_nav.dismiss_dialogs()
    print('\u2705 Ready to scan stories.')
else:
    print('\u274c Authentication failed. Check your session ID.')

## 7. Run the Full Pipeline

This cell runs the complete flow:
1. Discover stories from the tray
2. Capture frames from each story
3. Run OCR on captured frames
4. 4B screening (decide SUFFICIENT vs REVISIT)
5. Revisit queue processing
6. 8B detailed analysis
7. 32B expert review (if needed and available)

In [ ]:
from src.pipeline import StoryPipeline

pipeline = StoryPipeline(config, db)
pipeline.set_browser(session)
pipeline.set_navigators(instagram_nav, story_nav)
pipeline.set_sampler(sampler, frame_manager)
pipeline.set_models(screener, analyzer, expert)
pipeline.set_ocr(ocr_engine)

print('Starting pipeline...\n')
stats = await pipeline.run()

print(f'\n--- Pipeline Complete ---')
print(f'  Discovered: {stats["discovered"]}')
print(f'  Completed:  {stats["completed"]}')
print(f'  Revisited:  {stats["revisited"]}')
print(f'  Failed:     {stats["failed"]}')

## 8. View Results

In [ ]:
from src.analysis.report import ReportGenerator

report_gen = ReportGenerator(db)

# Text report
report = report_gen.generate_text_report()
if report:
    print(report)
else:
    print('No stories analyzed yet.')

## 9. Export Data

In [ ]:
import os
os.makedirs(config.data_dir, exist_ok=True)

json_path = config.data_dir / 'export.json'
csv_path = config.data_dir / 'export.csv'

report_gen.export_json(json_path)
report_gen.export_csv(csv_path)

print(f'\u2705 JSON exported to {json_path}')
print(f'\u2705 CSV exported to {csv_path}')

# Download files from Colab
try:
    from google.colab import files
    files.download(str(json_path))
    files.download(str(csv_path))
except ImportError:
    pass

## 10. Cleanup

In [ ]:
# Close browser
await session.close()
print('\u2705 Browser closed.')

# Unload models to free VRAM
screener.unload_model()
analyzer.unload_model()
if expert:
    expert.unload_model()
print('\u2705 Models unloaded.')

# Close database
db.close()
print('\u2705 Database closed.')